In [ ]:
from datetime import date
from itertools import chain
from pathlib import Path
import importlib
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import dep_ecp

from dep_ecp import ecp_v3_aggSec as ecp_sec_em
from dep_ecp import ecp_v3_coverageFactors as ecp_cov_fac
from dep_ecp import ecp_v3_gen_func as ecp_general
from dep_ecp import ecp_v3_overlap as ecp_overlap
from wcpd_utils import jurisdictions as jur_loader

importlib.reload(ecp_general)
importlib.reload(ecp_cov_fac)
importlib.reload(ecp_overlap)
importlib.reload(ecp_sec_em)
importlib.reload(dep_ecp)

today = date.today()
d1 = today.strftime("%b-%d-%Y")

def find_project_root(markers=("pyproject.toml", "setup.cfg", "requirements.txt", ".git", ".project-root")):
    path = Path.cwd().resolve()
    for parent in (path, *path.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent
    return path

REPO_ROOT = find_project_root()

def resolve_wcpd_repo_root():
    candidates = []
    env_root = os.environ.get("WCPD_REPO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        REPO_ROOT.parent / "WorldCarbonPricingDatabase",
        Path.home() / "GitHub" / "WorldCarbonPricingDatabase",
    ])
    for candidate in candidates:
        if (candidate / "_dataset" / "data").exists() and (candidate / "_raw" / "overlap").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the WorldCarbonPricingDatabase checkout. Set WCPD_REPO_ROOT to the repo root."
    )

def sync_wcpd_repo(repo_root):
    if os.environ.get("WCPD_SKIP_PULL") == "1":
        print("Skipping WCPD sync because WCPD_SKIP_PULL=1.")
        return
    if not (repo_root / ".git").exists():
        print(f"Skipping WCPD sync because {repo_root} is not a git checkout.")
        return
    status = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        print("Skipping WCPD sync because the checkout has local changes.")
        return
    result = subprocess.run(
        ["git", "-C", str(repo_root), "pull", "--ff-only"],
        check=True,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())

def resolve_wcpd_gas_path(gas):
    versions = [DB_VERSION] + DB_VERSION_FALLBACKS.get(gas, [])
    checked = []
    for version in versions:
        gas_path = WCPD_REPO_ROOT / "_dataset" / "data" / version / gas
        national = gas_path / "national"
        subnational = gas_path / "subnational"
        checked.append(str(gas_path))
        if national.exists() and subnational.exists() and any(national.glob("*.csv")) and any(subnational.glob("*.csv")):
            return gas_path, version
    raise FileNotFoundError(f"Could not find WCPD files for {gas}. Checked: {checked}")

WCPD_REPO_ROOT = resolve_wcpd_repo_root()
DB_VERSION = "v2026.1"
DB_VERSION_FALLBACKS = {"CH4": ["v2025.0"], "N2O": ["v2025.0"]}
lastDbYear = 2025
path_ghg_processed = REPO_ROOT / "_raw" / "ghg_inventory" / "processed"
path_aux_files = REPO_ROOT / "_raw" / "_aux_files"
path_dataset_output = REPO_ROOT / "_output" / "_dataset" / DB_VERSION

sync_wcpd_repo(WCPD_REPO_ROOT)

ecp_sec_em.WCPD_USD_ROOT = REPO_ROOT / "_raw" / "wcpd_usd"
ecp_sec_em.CF_WEIGHTED_OUTPUT_ROOT = REPO_ROOT / "_raw" / "wcpd_cfWeightedPrices_usd"
ecp_sec_em.inventoryPath = path_ghg_processed
ecp_sec_em.LAST_DB_YEAR = lastDbYear

jurisdictions = jur_loader.jurisdictions
countries = ["United States", "Canada", "China"]
missing = [country for country in countries if country not in jurisdictions.get("subnationals", {})]
if missing:
    raise KeyError(f"Missing subnational lists for: {missing}")

subnat_lists = {country: jurisdictions["subnationals"][country] for country in countries}
all_subnat_list = list(chain.from_iterable(subnat_lists[country] for country in countries))
gases = ["CO2"]


Loading data

In [ ]:
wcpd = {}

ipcc_iea_map = pd.read_csv(
    path_aux_files / "ipcc2006_iea_category_codes.csv",
    usecols=["ipcc_code", "FLOW"],
).rename(columns={"FLOW": "iea_code"})

for gas in gases:
    gas_path, gas_version = resolve_wcpd_gas_path(gas)
    print(f"Building policy features data frame for {gas} from {gas_version}")

    wcpd_ctry = ecp_general.concatenate(str(gas_path / "national"))
    wcpd_subnat = ecp_general.concatenate(str(gas_path / "subnational"))
    wcpd_all = pd.concat([wcpd_ctry, wcpd_subnat], ignore_index=True).sort_values(by=["jurisdiction", "year"])

    wcpd_all["Product"] = wcpd_all["Product"].fillna("NA")
    wcpd_all = wcpd_all.drop_duplicates(subset=["jurisdiction", "year", "ipcc_code", "Product"])
    wcpd_all = wcpd_all.merge(ipcc_iea_map, on="ipcc_code", how="left")
    wcpd_all["iea_code"] = wcpd_all["iea_code"].fillna("NA")

    cols = ["ets_2_id", "tax_2_id", "ets_2_curr_code"]
    wcpd_all = wcpd_all.astype({column: "string" for column in cols if column in wcpd_all.columns})

    def standardize(names):
        return {name: name.replace(".", "").replace(",", "").replace(" ", "_") for name in names}

    ctry_names = wcpd_ctry["jurisdiction"].unique()
    subnat_names = wcpd_subnat["jurisdiction"].unique()
    countries_dic = standardize(ctry_names)
    subnat_dic = standardize(subnat_names)

    if wcpd_all.duplicated(["jurisdiction", "year", "ipcc_code", "Product"]).any():
        print(f"The dataset for {gas} contains duplicates!")

    wcpd_all = ecp_cov_fac.coverageFactors(wcpd_all, gas, WCPD_REPO_ROOT)
    overlap = pd.read_csv(WCPD_REPO_ROOT / "_raw" / "overlap" / f"overlap_mechanisms_{gas}.csv")
    wcpd_all = ecp_overlap.overlap(wcpd_all, overlap)
    wcpd[gas] = wcpd_all


In [ ]:
priceSeriesPaths = {
    "cFlxRate": "currentPrices/FlexXRate",
    "cFixRate": "currentPrices/FixedXRate",
    "kFixRate": "constantPrices/FixedXRate",
}

price_cols = {
    "cFlxRate": ["ets_price_usd", "tax_rate_incl_ex_usd"],
    "cFixRate": ["ets_price_usd", "tax_rate_incl_ex_usd"],
    "kFixRate": ["ets_price_usd_k", "tax_rate_incl_ex_usd_k"],
}

ecp_cols = {
    "cFlxRate": ["ecp_ets_usd", "ecp_tax_usd", "ecp_all_usd"],
    "cFixRate": ["ecp_ets_usd", "ecp_tax_usd", "ecp_all_usd"],
    "kFixRate": ["ecp_ets_usd_k", "ecp_tax_usd_k", "ecp_all_usd_k"],
}

IPCC1AList = [
    "1A", "1B", "1C", "1A1", "1A2", "1A3", "1A4", "1A5", "1A1A", "1A1B", "1A1C", "1A2A", "1A2B",
    "1A2C", "1A2D", "1A2E", "1A2F", "1A2G", "1A2H", "1A2I", "1A2J", "1A2K", "1A2L", "1A2M", "1A3A",
    "1A3B", "1A3C", "1A3D", "1A3E", "1A4A", "1A4B", "1A4C", "1A5A", "1A5B", "1A5C", "1A1A1", "1A1A2",
    "1A1A3", "1A3A1", "1A3A2", "1A3D1", "1A3D2", "1A3E1", "1A4C1", "1A4C2", "1A4C3", "1A5A", "1A5B",
    "1A5C",
]

priceSeriesList = ["cFlxRate", "kFixRate"]
jurGroup = "national"

priceCat = {
    "level_5": ["1A1A1", "1A1A2", "1A1A3", "1A3A1", "1A3A2", "1A3D1", "1A3D2", "1A3E1", "1A4C1", "1A4C2", "1A4C3"],
    "level_4": ["1A1A", "1A1B", "1A1C", "1A2A", "1A2B", "1A2C", "1A2D", "1A2E", "1A2F", "1A2G", "1A2H", "1A2I", "1A2J", "1A2K", "1A2L", "1A2M", "1A3A", "1A3B", "1A3C", "1A3D", "1A4A", "1A4B", "1A4C", "1A5A", "1A5B", "1A5C", "1B2A", "1B2B", "3C1A", "3C1B", "3C1C", "3C1D"],
    "level_3": ["1A2", "1A5", "1B1", "2A1", "2A2", "2A3", "2A4", "2H1", "2H2", "3A1", "3A2", "3B1", "3B2", "3B3", "3B4", "3B5", "3B6", "3C1", "3C2", "3C3", "3C4", "3C5", "3C6", "3C7", "3C8"],
    "level_2": ["2A", "2B", "2C", "2D", "2E", "2F", "2G", "4A", "4B", "4C", "4D", "4E", "5A", "5B"],
}

aggCatList = [
    "1", "1A", "1A1A", "1A2", "1A3", "1A3A", "1A3D", "1A3E", "1A4", "1A4C", "1A5", "1B", "1B1", "1B1A",
    "1B1A1", "1B1A2", "1B2", "1B2A", "1B2A3", "1B2B", "1B2B3", "1C", "1C1", "1C2", "2", "2A", "2A4",
    "2B", "2B8", "2B9", "2C", "2D", "2E", "2F", "2G", "2H", "3", "3A", "3B", "3B1", "3B2", "3B3",
    "3B4", "3B5", "3B6", "3B6B", "3C", "3D", "4", "4A", "4C", "4D", "5", "5A",
]

IPCC1AListSubCat = list(set(IPCC1AList) - set(aggCatList))


In [ ]:
dfSecPrice = {}
sector_price_inputs = {}

for gas in gases:
    dfSecPrice[gas] = {}
    sector_price_inputs[gas] = {}

    for priceSeries in priceSeriesList:
        results = []
        priceSeriesPath = priceSeriesPaths[priceSeries]
        cfWprices_usd, all_inst_col = ecp_sec_em.cfWeightedPrices(
            gas,
            priceSeries,
            priceSeriesPath,
            price_cols,
            wcpd[gas],
            countries_dic,
            subnat_dic,
        )
        sector_price_inputs[gas][priceSeries] = cfWprices_usd.copy()

        for level, categories in priceCat.items():
            for category in categories:
                inventoryShare = ecp_sec_em.inventoryShare(category, jurGroup, gas, level)
                merge_keys = ["jurisdiction", "year", "ipcc_code", "iea_code"]
                if category in IPCC1AListSubCat and jurGroup == "national":
                    merge_keys.append("Product")

                temp = inventoryShare.merge(cfWprices_usd, on=merge_keys, how="left")
                share_col = f"{gas}_shareAggSec"

                if category in IPCC1AListSubCat and jurGroup == "national":
                    total_emissions = temp.groupby(["jurisdiction", "year"])[gas].transform("sum")
                    temp.loc[temp[share_col].isna() & (total_emissions == 0), share_col] = 1 / 3
                    temp.loc[temp[share_col].isna() & (total_emissions != 0), share_col] = 0
                else:
                    subCatcodes = [
                        code for code in temp.ipcc_code.unique()
                        if code.startswith(category) and len(code) == len(category) + 1
                    ]
                    weight = 1 / len(subCatcodes) if subCatcodes else 1
                    cond = temp.ipcc_code.isin(subCatcodes) if subCatcodes else temp.ipcc_code == category
                    temp.loc[cond & temp[share_col].isna(), share_col] = weight

                temp = temp.fillna({price_cols[priceSeries][0]: 0, price_cols[priceSeries][1]: 0})
                temp = temp.assign(
                    **{
                        ecp_cols[priceSeries][0]: temp[price_cols[priceSeries][0]] * temp[share_col],
                        ecp_cols[priceSeries][1]: temp[price_cols[priceSeries][1]] * temp[share_col],
                    }
                )
                temp[ecp_cols[priceSeries][2]] = temp[ecp_cols[priceSeries][0]] + temp[ecp_cols[priceSeries][1]]
                temp = temp.drop(columns=[all_inst_col] + price_cols[priceSeries])

                if level == "level_5":
                    group_cols = ["jurisdiction", "year", "ipcc_code"]
                else:
                    if temp.ipcc_code.nunique() > 1:
                        temp = temp[temp.ipcc_code != category]
                    group_cols = ["jurisdiction", "year"]

                temp_sum = temp.groupby(group_cols, as_index=False).sum(numeric_only=True)
                if level != "level_5":
                    temp_sum["ipcc_code"] = category

                results.append(temp_sum)

                if category in aggCatList:
                    cfWprices_usd = cfWprices_usd[cfWprices_usd.ipcc_code != category]
                    cols = ["jurisdiction", "year", "ipcc_code"] + ecp_cols[priceSeries]
                    temp_agg = temp_sum[cols].assign(Product=np.nan)
                    colMap = dict(zip(ecp_cols[priceSeries], price_cols[priceSeries] + [all_inst_col]))
                    cfWprices_usd = pd.concat([cfWprices_usd, temp_agg.rename(columns=colMap)], ignore_index=True)

        if not results:
            raise ValueError(f"No sector ECP results were generated for {gas} {priceSeries}")

        df_final = pd.concat(results, ignore_index=True).sort_values(["jurisdiction", "year", "ipcc_code"])
        output_dir = path_dataset_output / "ecp" / "ipcc" / "ecp_ipcc" / Path(priceSeriesPath)
        output_dir.mkdir(parents=True, exist_ok=True)
        df_final.fillna("NA").to_csv(output_dir / f"ecp_ipcc_{gas}_{priceSeries}.csv", index=False)
        dfSecPrice[gas][priceSeries] = df_final


Constant (year of introduction), jurisdiction-specific, weights

**Categories 1A (combustion) only**

In [ ]:
intro_gas = gases[0]
firstYear = wcpd[intro_gas][["jurisdiction", "year", "ipcc_code", "iea_code", "Product", "tax", "ets"]].copy()
firstYear["pricing"] = firstYear["tax"] + firstYear["ets"]
firstYear["pricing"] = np.where(firstYear["pricing"] > 0, 1.0, 0.0)
firstYear = firstYear.drop(columns=["tax", "ets"])
firstYear = firstYear.loc[firstYear["pricing"] == 1].copy()
firstYear = firstYear.sort_values(by=["jurisdiction", "year", "ipcc_code", "Product"])
firstYear = firstYear.drop_duplicates(subset=["jurisdiction", "ipcc_code", "Product"])

firstYear_cat = firstYear.groupby(["jurisdiction", "year", "ipcc_code", "iea_code"], as_index=False).sum(numeric_only=True)
firstYear_cat["pricing"] = np.where(firstYear_cat["pricing"] > 0, 1.0, 0.0)
firstYear_cat = firstYear_cat.drop_duplicates(subset=["jurisdiction", "iea_code"])
firstYear_cat["year"] = firstYear_cat["year"] - 1
firstYear_cat = firstYear_cat.drop(columns=["pricing"])
firstYear_cat.loc[(firstYear_cat.jurisdiction == "Finland") & (firstYear_cat.year == 1989), "year"] = 1990
firstYear_cat.loc[(firstYear_cat.jurisdiction == "Poland") & (firstYear_cat.year == 1989), "year"] = 1990


In [ ]:
invName = {"national": "nat", "subnational": "subnat"}
inventory = pd.read_csv(path_ghg_processed / f"inventory_{invName[jurGroup]}_{intro_gas}.csv")
inventory = inventory[["jurisdiction", "year", "ipcc_code", "iea_code", "Product", intro_gas]].copy()

last_inventory_year = int(inventory["year"].max())
if last_inventory_year < lastDbYear:
    latest_inventory = inventory.loc[inventory["year"] == last_inventory_year].copy()
    for year in range(last_inventory_year + 1, lastDbYear + 1):
        inventory = pd.concat([inventory, latest_inventory.assign(year=year)], ignore_index=True)

inventory = inventory.loc[inventory["iea_code"].notna()].copy()
aggSecEm = inventory.groupby(["jurisdiction", "year", "ipcc_code", "iea_code"], as_index=False).sum(numeric_only=True)

share_df = inventory.merge(
    aggSecEm[["jurisdiction", "year", "iea_code", intro_gas]],
    on=["jurisdiction", "year", "iea_code"],
    how="left",
    suffixes=("", "_agg"),
)
share_df[f"{intro_gas}_shareAggSec"] = share_df[intro_gas] / share_df[f"{intro_gas}_agg"]
share_df = share_df.drop(columns=[f"{intro_gas}_agg"])


In [ ]:
def ecp_constIntroCat(share_df, prices, gas):
    frames = []
    first_year_lookup = firstYear_cat.set_index(["jurisdiction", "iea_code"])["year"]
    available_lookup = share_df.groupby(["jurisdiction", "iea_code"])["year"].max()
    share_col = f"{gas}_shareAggSec"

    for jur in share_df["jurisdiction"].dropna().unique():
        jur_df = share_df.loc[share_df["jurisdiction"] == jur]
        for sector in jur_df["iea_code"].dropna().unique():
            key = (jur, sector)
            if key in first_year_lookup.index:
                weight_year = int(first_year_lookup.loc[key])
            else:
                weight_year = 2015

            latest_available_year = int(available_lookup.loc[key])
            weight_year = min(weight_year, latest_available_year)

            temp_df = jur_df.loc[(jur_df["year"] == weight_year) & (jur_df["iea_code"] == sector)].copy()
            if temp_df.empty:
                temp_df = jur_df.loc[(jur_df["year"] == latest_available_year) & (jur_df["iea_code"] == sector)].copy()
            if temp_df.empty:
                continue

            temp_df = temp_df.drop(columns=["year"])
            merge_keys = ["jurisdiction", "ipcc_code", "iea_code", "Product"]
            temp_df = temp_df.merge(prices, on=merge_keys, how="left")
            temp_df[["ets_price_usd_k", "tax_rate_incl_ex_usd_k"]] = temp_df[["ets_price_usd_k", "tax_rate_incl_ex_usd_k"]].fillna(0)

            temp_df["ecp_ets_ew_usd_k"] = temp_df["ets_price_usd_k"] * temp_df[share_col]
            temp_df["ecp_tax_ew_usd_k"] = temp_df["tax_rate_incl_ex_usd_k"] * temp_df[share_col]
            temp_df["ecp_all_ew_usd_k"] = (temp_df["ets_price_usd_k"] + temp_df["tax_rate_incl_ex_usd_k"]) * temp_df[share_col]

            temp_df = temp_df.drop(columns=["ets_price_usd_k", "tax_rate_incl_ex_usd_k", "all_inst_usd_k"])
            temp_df_sum = temp_df.groupby(["jurisdiction", "year", "iea_code"], as_index=False).sum(numeric_only=True)
            frames.append(
                temp_df_sum[[
                    "jurisdiction",
                    "year",
                    "iea_code",
                    "ecp_ets_ew_usd_k",
                    "ecp_tax_ew_usd_k",
                    "ecp_all_ew_usd_k",
                ]]
            )

    if not frames:
        raise ValueError("No constant-introduction sector ECP data were generated.")

    return pd.concat(frames, ignore_index=True)


In [ ]:
ecp_ipcc_intro = ecp_constIntroCat(share_df, sector_price_inputs[intro_gas]["kFixRate"], intro_gas)


In [ ]:
intro_output_dir = path_dataset_output / "ecp" / "ipcc" / "ecp_intro"
intro_output_dir.mkdir(parents=True, exist_ok=True)
ecp_ipcc_intro.to_csv(intro_output_dir / f"ecp_ipcc_{intro_gas}_intro.csv", index=False)
